# From Scratch

Base Block

In [25]:
import numpy as np
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [26]:
import torch
import torch.nn as nn
class BasicBlock(nn.Module):
  def __init__(self,in_channels,out_channels,stride=1):
    super().__init__()
    self.conv1=nn.Conv2d(in_channels=in_channels,out_channels=out_channels,kernel_size=3,
                         stride=stride,padding=1,bias=False)
    self.bn1=nn.BatchNorm2d(out_channels)
    self.conv2=nn.Conv2d(in_channels=out_channels,out_channels=out_channels,kernel_size=3,stride=1,padding=1,bias=False)
    self.bn2=nn.BatchNorm2d(out_channels)
    self.shortcut=nn.Sequential()
    #If dimension change then do projection
    if stride!=1 or in_channels !=out_channels:
      self.shortcut=nn.Sequential(
          nn.Conv2d(in_channels=in_channels,out_channels=out_channels,kernel_size=1,stride=stride,bias=True),
          nn.BatchNorm2d(out_channels)
      )

  def forward(self,x):
    out=torch.relu(self.bn1(self.conv1(x)))
    out=self.bn2(self.conv2(out))
    out+=self.shortcut(x)
    out=torch.relu(out)
    return out

ResNet

In [27]:
class ResNet(nn.Module):
  def __init__(self,block,layers,num_classes=10):
    super().__init__()
    self.in_channels=64
    self.conv1=nn.Conv2d(in_channels=3,out_channels=64,kernel_size=7,stride=2,padding=3)
    self.bn1=nn.BatchNorm2d(64)
    # self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

    self.layer1=self.make_layer(block,64,layers[0])
    self.layer2=self.make_layer(block,128,layers[1],stride=2)
    self.layer3=self.make_layer(block,256,layers[2],stride=2)
    self.layer4=self.make_layer(block,512,layers[3],stride=2)

    self.avgpool=nn.AdaptiveAvgPool2d((1,1))
    self.fc=nn.Linear(512,num_classes)

  def make_layer(self,block,out_channels,blocks,stride=1):
    layers=[]
    layers.append(block(self.in_channels,out_channels,stride))
    self.in_channels=out_channels

    for _ in range(1,blocks):
      layers.append(block(self.in_channels,out_channels))
    return nn.Sequential(*layers)

  def forward(self,x):
    x=torch.relu(self.bn1(self.conv1(x)))
    # x=self.maxpool(x)
    x=self.layer1(x)
    x=self.layer2(x)
    x=self.layer3(x)
    x=self.layer4(x)
    x=self.avgpool(x)
    x=torch.flatten(x,1)
    x=self.fc(x)
    return x

In [28]:
model=ResNet(BasicBlock,[2,2,2,2],num_classes=10)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)

In [29]:
scheduler=torch.optim.lr_scheduler.StepLR(optimizer,step_size=5,gamma=0.5)

In [30]:
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
train_transform=transforms.Compose([
    transforms.Resize(36),
    transforms.RandomCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.228,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(36),
    transforms.CenterCrop(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root='./data',download=True,train=True,transform=train_transform)
test_data=datasets.CIFAR10(root='./data',download=True,train=False,transform=test_transform)

train_loader=DataLoader(train_data,batch_size=128,shuffle=True,pin_memory=True,num_workers=2)
test_loader=DataLoader(test_data,batch_size=128,shuffle=False,pin_memory=True,num_workers=2)

In [31]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)

In [32]:
EPOCHS=10
for epoch in range(EPOCHS):
  model.train()
  total_loss=0
  for images,labels in train_loader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output=model(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  # Update the learning rate scheduler
  scheduler.step()
  print(f"The loss per batch for epoch: {epoch+1}/{EPOCHS} is {total_loss/len(train_loader)}")

The loss per batch for epoch: 1/10 is 2.0459363698349584
The loss per batch for epoch: 2/10 is 1.4805874894647038
The loss per batch for epoch: 3/10 is 1.2772822639216548
The loss per batch for epoch: 4/10 is 1.0979261625453334
The loss per batch for epoch: 5/10 is 0.9655173939207325
The loss per batch for epoch: 6/10 is 0.7696919384819773
The loss per batch for epoch: 7/10 is 0.7036537591301267
The loss per batch for epoch: 8/10 is 0.6553877662972111
The loss per batch for epoch: 9/10 is 0.6147637255203998
The loss per batch for epoch: 10/10 is 0.5733953784493839


In [33]:
model.eval()
correct=0
total=0
with torch.no_grad():
  for images,labels in test_loader:
    images,labels=images.to(device),labels.to(device)
    output=model(images)
    _,predicted=torch.max(output,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()
    accuracy=100*correct/total
print(f"Epoch [{epoch+1}/{EPOCHS}], Test Accuracy: {accuracy:.2f}%")

Epoch [10/10], Test Accuracy: 78.11%


# Transfer Learning

In [34]:
train_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
test_transform=transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
])
train_data=datasets.CIFAR10(root='./data',download=True,train=True,transform=train_transform)
test_data=datasets.CIFAR10(root='./data',download=True,train=False,transform=test_transform)

train_loader=DataLoader(train_data,batch_size=128,shuffle=True,pin_memory=True,num_workers=2)
test_loader=DataLoader(test_data,batch_size=128,shuffle=False,pin_memory=True,num_workers=2)

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
model=models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 157MB/s]


In [36]:
for param in model.parameters():
  param.requires_grad=False

In [38]:
model.fc=nn.Linear(in_features=model.fc.in_features,out_features=10)

In [39]:
model=model.to(device)

In [40]:
optimizer=optim.SGD(model.fc.parameters(),lr=0.01,momentum=0.9)

In [41]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)
        correct += (predicted==labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")

Epoch [1/10]
Train Loss: 0.8524, Train Acc: 70.48%
Epoch [2/10]
Train Loss: 0.7068, Train Acc: 75.50%
Epoch [3/10]
Train Loss: 0.6842, Train Acc: 76.25%
Epoch [4/10]
Train Loss: 0.6767, Train Acc: 76.54%
Epoch [5/10]
Train Loss: 0.6691, Train Acc: 76.72%
Epoch [6/10]
Train Loss: 0.6705, Train Acc: 76.88%
Epoch [7/10]
Train Loss: 0.6598, Train Acc: 77.23%
Epoch [8/10]
Train Loss: 0.6615, Train Acc: 77.29%
Epoch [9/10]
Train Loss: 0.6609, Train Acc: 76.99%
Epoch [10/10]
Train Loss: 0.6611, Train Acc: 77.34%


In [43]:
model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    test_loss += loss.item()
    _, predicted = outputs.max(1)
    correct += predicted.eq(labels).sum().item()
    total += labels.size(0)
  test_acc = 100 * correct / total

print(f"Epoch [{epoch+1}/{EPOCHS}]")
print(f"Test Loss: {test_loss/len(test_loader):.4f}, Test Acc: {test_acc:.2f}%")

Epoch [10/10]
Test Loss: 0.6432, Test Acc: 78.49%


In [44]:
for name,param in model.named_parameters():
  if "layer4" in name:
    param.requires_grad=True

In [45]:
model=model.to(device)
optimizer=optim.SGD([
    {'params':model.fc.parameters(),'lr':0.01},
    {'params':model.layer4.parameters(),'lr':0.001}
    ],momentum=0.9)

In [46]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)
        correct += (predicted==labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")

Epoch [1/5]
Train Loss: 0.5381, Train Acc: 81.51%
Epoch [2/5]
Train Loss: 0.4118, Train Acc: 85.61%
Epoch [3/5]
Train Loss: 0.3547, Train Acc: 87.70%
Epoch [4/5]
Train Loss: 0.3097, Train Acc: 89.04%
Epoch [5/5]
Train Loss: 0.2811, Train Acc: 90.08%


In [47]:
model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    test_loss += loss.item()
    _, predicted = outputs.max(1)
    correct += predicted.eq(labels).sum().item()
    total += labels.size(0)
  test_acc = 100 * correct / total

print(f"Epoch [{epoch+1}/{EPOCHS}]")
print(f"Test Loss: {test_loss/len(test_loader):.4f}, Test Acc: {test_acc:.2f}%")

Epoch [5/5]
Test Loss: 0.3373, Test Acc: 88.59%
